Integrante: Bastian Contreras

Programa 1 (Generador Encuesta)

In [2]:
import random

# Creamos un bucle While para poder hacer el menú de ingreso de filas
# y agregamos un 'Try' para evitar errores como por ejemplo ingresar palabras
while True:
    try:
        numero_filas = int(input('Ingrese el número de filas deseado (mínimo 50): '))
        if numero_filas >= 50:
            break  # Con esto podemos salir del bucle una vez que el usuario coloque un número válido
        else:
            print('Por favor ingrese un número de filas mayor o igual a 50.')
    except ValueError:
        print('¡Error! Por favor ingrese un número válido.')

# Listas posibles de valores
sexos = ['Masculino', 'Femenino']
respuestas = ['Si', 'No', 'Tal vez']

# Ahora generamos un archivo CSV
with open('Encuesta_habitos.csv', 'w') as archivo:  # Declaramos 'archivo'
    archivo.write('Sexo,Edad,Respuesta\n')  # Escribimos el encabezado con comas

    for i in range(numero_filas):
        sexo = random.choice(sexos)
        edad = random.randint(18, 45)
        respuesta = random.choice(respuestas)
        archivo.write(f'{sexo},{edad},{respuesta}\n')
print(f'Archivo creado con exito con: {numero_filas} filas')

Archivo creado con exito con: 80 filas


Programa 2 (procesador encuesta)

In [ ]:
# Archivo que quiero usar para trabajar, tiene que llamarse exactamente así
archivo_correcto = 'Encuesta_habitos.csv'

# Acá voy a ir guardando los datos que saco del archivo
datos = {'Sexo': [], 'Edad': [], 'Respuesta': []}

# Para convertir las respuestas tipo texto a números que pueda procesar más fácil
respuesta_a_numeros = {'Si': 1, 'No': 3, 'Tal vez': 2}

# Solo muestra el menú con las opciones disponibles
def mostrar_menu():
    print ("\n")
    print(f'---Menu---')
    print('Seleccione una opción (Debe ingresar el archivo a evaluar en la opción 1)')
    print(f'1. Leer archivo de datos')
    print(f'2. Mostrar estadísticas generales (media, mediana, conteo de respuestas por tipo)')
    print(f'3. Filtrar datos por sexo y mostrar estadísticas')
    print(f'4. Filtrar datos por rango de edad y mostrar estadísticas')
    print(f'5. Guardar resultados en un archivo')
    print(f'6. Salir')

# Esta función se encarga de cargar el archivo
def leer_archivo():
    nombre_archivo = input('Ingrese el nombre del archivo: ')
    
    # Me aseguro de que sea el archivo correcto
    if nombre_archivo != archivo_correcto:
        print('Error! Archivo no encontrado. Por favor ingrese nuevamente.')
        return

    try:
        with open(nombre_archivo, 'r') as archivo:
            print(f"El archivo '{nombre_archivo}' ha sido cargado exitosamente.")

            leer_lineas = archivo.readlines()

            # Borro los datos anteriores por si ya se había cargado algo antes
            datos['Sexo'].clear()
            datos['Edad'].clear()
            datos['Respuesta'].clear()

            # Empiezo desde la segunda línea para evitar el encabezado
            for linea in leer_lineas[1:]:
                columna = linea.strip().split(',')

                # A veces puede venir una línea con datos faltantes, la salto
                if len(columna) < 3:
                    print(f'Linea de datos incompletos --> {linea.strip()}')
                    continue

                respuesta = columna[2].strip()

                # Este caso lo agregué por si en vez de "Tal vez" ponen solo "Tal"
                if respuesta.lower() == "tal":
                    respuesta = "Tal vez"

                # Busco el valor numérico de la respuesta
                respuesta_numerica = respuesta_a_numeros.get(respuesta, 0)

                try:
                    datos['Sexo'].append(columna[0].strip())
                    datos['Edad'].append(int(columna[1].strip()))
                    datos['Respuesta'].append(respuesta_numerica)
                except ValueError:
                    # Si la edad no es un número, aviso el error
                    print(f"Error en la línea: {linea.strip()} (Edad no válida)")

            print('Datos cargados correctamente.')

    except FileNotFoundError:
        print('Error! Archivo no encontrado. Por favor ingrese nuevamente.')

# Muestra estadísticas básicas con los datos cargados
def estadisticas_generales():
    if not datos['Edad']:
        print('No se han cargado los datos. Por favor intente de nuevo.')
        return

    # Calculo el promedio de edad
    promedio_edad = sum(datos['Edad']) / len(datos['Edad'])
    print(f'El promedio de edades es: {promedio_edad:.2f}')

    # Ahora la mediana, que es un poco más manual
    edades_ordenadas = sorted(datos['Edad'])
    cantidad_datos = len(edades_ordenadas)

    if cantidad_datos % 2 == 1:
        mediana = edades_ordenadas[cantidad_datos // 2]
    else:
        mediana = (edades_ordenadas[cantidad_datos // 2 - 1] + edades_ordenadas[cantidad_datos // 2]) / 2

    print(f'La mediana de edad es: {mediana}')

    # Recuento de cuántas veces se respondió "Sí", "No" o "Tal vez"
    contados_datos = {'Si': 0, 'Tal vez': 0, 'No': 0}
    for respuesta in datos['Respuesta']:
        if respuesta == 1:
            contados_datos['Si'] += 1
        elif respuesta == 2:
            contados_datos['Tal vez'] += 1
        elif respuesta == 3:
            contados_datos['No'] += 1

    print('--Cantidad de respuestas--')
    print(f'Si: {contados_datos["Si"]}')
    print(f'Tal vez: {contados_datos["Tal vez"]}')
    print(f'No: {contados_datos["No"]}')

# Filtra los datos según el sexo que ingrese el usuario
def filtro_datos_por_sexo():
    sexo_filtro = input("Ingrese el sexo a filtrar (Masculino/Femenino): ").strip()
    cantidad = datos['Sexo'].count(sexo_filtro)

    if cantidad == 0:
        print(f"No hay datos para el sexo '{sexo_filtro}'.")
    else:
        print(f"Cantidad de respuestas para '{sexo_filtro}': {cantidad}")

# Esta función permite filtrar por un rango de edad específico
def filtro_datos_por_rango_edad():
    try:
        edad_min = int(input("Ingrese edad mínima: "))
        edad_max = int(input("Ingrese edad máxima: "))
    except ValueError:
        print("Debe ingresar números válidos.")
        return

    edades_filtradas = []
    for i in range(len(datos['Edad'])):
        edad = datos['Edad'][i]
        if edad_min <= edad <= edad_max:
            edades_filtradas.append(edad)

    if not edades_filtradas:
        print(f"No hay datos en el rango de edad {edad_min}-{edad_max}.")
    else:
        promedio = sum(edades_filtradas) / len(edades_filtradas)
        print(f"Cantidad de personas en el rango: {len(edades_filtradas)}")
        print(f"Promedio de edad en el rango: {promedio:.2f}")

        # Y la mediana del rango también
        edades_filtradas.sort()
        cantidad_datos = len(edades_filtradas)
        if cantidad_datos % 2 == 1:
            mediana = edades_filtradas[cantidad_datos // 2]
        else:
            mediana = (edades_filtradas[cantidad_datos // 2 - 1] + edades_filtradas[cantidad_datos // 2]) / 2

        print(f"Mediana de edad en el rango: {mediana}")

# Esta parte guarda todo lo que se analizó en un archivo de texto
def guardar_resultados():
    nombre_archivo = input("Ingrese nombre para guardar los resultados (ej: resultados.txt): ")
    
    try:
        with open(nombre_archivo, 'w') as archivo:
            archivo.write("Resultados de la encuesta:\n")
            archivo.write(f"Total de registros: {len(datos['Edad'])}\n")
            
            promedio = sum(datos['Edad']) / len(datos['Edad'])
            archivo.write(f"Promedio de edad: {promedio:.2f}\n")
            
            # Vuelvo a contar respuestas para dejarlas guardadas también
            respuestas = {'Si': 0, 'Tal vez': 0, 'No': 0}
            for r in datos['Respuesta']:
                if r == 1:
                    respuestas['Si'] += 1
                elif r == 2:
                    respuestas['Tal vez'] += 1
                elif r == 3:
                    respuestas['No'] += 1

            archivo.write("Respuestas:\n")
            archivo.write(f"Si: {respuestas['Si']}\n")
            archivo.write(f"Tal vez: {respuestas['Tal vez']}\n")
            archivo.write(f"No: {respuestas['No']}\n")
        
        print(f"Resultados guardados en '{nombre_archivo}' correctamente.")
    except:
        print("Ocurrió un error al guardar el archivo.")

# Menú principal, lo que mantiene todo corriendo
def menu():
    while True:
        mostrar_menu()
        opcion = input('Seleccione una opción (1-6): ')

        if opcion == '1':
            print('\nOpción 1')
            leer_archivo()
        elif opcion == '2':
            print('\nOpción 2')
            estadisticas_generales()
        elif opcion == '3':
            print('\nOpción 3')
            filtro_datos_por_sexo()
        elif opcion == '4':
            print('\nOpción 4')
            filtro_datos_por_rango_edad()
        elif opcion == '5':
            print('\nOpción 5')
            guardar_resultados()
        elif opcion == '6':
            print('\nOpción 6')
            print("Saliendo del programa...")
            break
        else:
            print("\nOpción no válida. Intente de nuevo.")

# Arranco el programa desde acá
menu()




---Menu---
Seleccione una opción (Debe ingresar el archivo a evaluar en la opción 1)
1. Leer archivo de datos
2. Mostrar estadísticas generales (media, mediana, conteo de respuestas por tipo)
3. Filtrar datos por sexo y mostrar estadísticas
4. Filtrar datos por rango de edad y mostrar estadísticas
5. Guardar resultados en un archivo
6. Salir

Opción 1
El archivo 'Encuesta_habitos.csv' ha sido cargado exitosamente.
Datos cargados correctamente.


---Menu---
Seleccione una opción (Debe ingresar el archivo a evaluar en la opción 1)
1. Leer archivo de datos
2. Mostrar estadísticas generales (media, mediana, conteo de respuestas por tipo)
3. Filtrar datos por sexo y mostrar estadísticas
4. Filtrar datos por rango de edad y mostrar estadísticas
5. Guardar resultados en un archivo
6. Salir

Opción 2
El promedio de edades es: 32.60
La mediana de edad es: 32.0
--Cantidad de respuestas--
Si: 24
Tal vez: 30
No: 26


---Menu---
Seleccione una opción (Debe ingresar el archivo a evaluar en la opci

### Creación de la Encuesta

A continuación se presenta la captura de pantalla relacionada con la creación de la encuesta y su estructura.

![Creación de la Encuesta](Capturas%20de%20resultados/7.png)


### Cargado de archivo encuesta


![Resultado de la Encuesta](Capturas%20de%20resultados/1.png)



### Estadisticas - Parte 2

Se muestran los resultados estadisticos, promedio (media) y mediana.

![Resultado de la Encuesta - Análisis 2](Capturas%20de%20resultados/2.png)


### Filtrado por sexo - Parte 3

Se pidio un filtrado con el sexo Femeninino 

![Resultado de la Encuesta - Análisis 3](Capturas%20de%20resultados/3.png)


### Filtrado por rango de edad - Parte 4

Se aplicó en un rango de 18-25 años

![Resultado de la Encuesta - Análisis 4](Capturas%20de%20resultados/4.png)


### Guardado en un archivo - Parte 5
 

![Resultado de la Encuesta - Análisis 5](Capturas%20de%20resultados/5.png)


### Cierre de programa - Parte 6



![Resultado de la Encuesta - Análisis 6](Capturas%20de%20resultados/6.png)


In [ ]:
import csv
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# --- Cargar datos del CSV ---
sexos, edades, respuestas = [], [], []

with open('Encuesta_habitos.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        sexos.append(row['Sexo'].strip())
        edades.append(int(row['Edad'].strip()))
        respuestas.append(row['Respuesta'].strip())

# --- Conteos ---
conteo_respuestas = {'Si': respuestas.count('Si'), 'Tal vez': respuestas.count('Tal vez'), 'No': respuestas.count('No')}
conteo_sexo = {'Masculino': sexos.count('Masculino'), 'Femenino': sexos.count('Femenino')}

resp_por_sexo = {}
for sexo in ['Masculino', 'Femenino']:
    indices = [i for i, s in enumerate(sexos) if s == sexo]
    resp_por_sexo[sexo] = {
        'Si':      sum(1 for i in indices if respuestas[i] == 'Si'),
        'Tal vez': sum(1 for i in indices if respuestas[i] == 'Tal vez'),
        'No':      sum(1 for i in indices if respuestas[i] == 'No'),
    }

# --- Layout ---
fig = plt.figure(figsize=(14, 10))
fig.suptitle('Análisis de Encuesta de Hábitos', fontsize=16, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

colores_resp = ['#4CAF50', '#FFC107', '#F44336']
colores_sexo = ['#42A5F5', '#EC407A']

# 1. Barras: distribución de respuestas
ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.bar(conteo_respuestas.keys(), conteo_respuestas.values(), color=colores_resp, edgecolor='white')
ax1.set_title('Distribución de Respuestas')
ax1.set_ylabel('Cantidad')
ax1.set_ylim(0, max(conteo_respuestas.values()) + 10)
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')

# 2. Torta: distribución por sexo
ax2 = fig.add_subplot(gs[0, 1])
ax2.pie(conteo_sexo.values(), labels=conteo_sexo.keys(), autopct='%1.1f%%',
        colors=colores_sexo, startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax2.set_title('Distribución por Sexo')

# 3. Histograma: distribución de edades
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(edades, bins=range(18, 47), color='#7E57C2', edgecolor='white', alpha=0.85)
ax3.axvline(sum(edades)/len(edades), color='red', linestyle='--', linewidth=1.5, label=f'Promedio: {sum(edades)/len(edades):.1f}')
ax3.set_title('Distribución de Edades')
ax3.set_xlabel('Edad')
ax3.set_ylabel('Frecuencia')
ax3.legend()

# 4. Barras agrupadas: respuestas por sexo
ax4 = fig.add_subplot(gs[1, 1])
categorias = ['Si', 'Tal vez', 'No']
x = range(len(categorias))
width = 0.35
bars_m = ax4.bar([i - width/2 for i in x], [resp_por_sexo['Masculino'][r] for r in categorias],
                 width, label='Masculino', color='#42A5F5', edgecolor='white')
bars_f = ax4.bar([i + width/2 for i in x], [resp_por_sexo['Femenino'][r] for r in categorias],
                 width, label='Femenino', color='#EC407A', edgecolor='white')
ax4.set_title('Respuestas por Sexo')
ax4.set_ylabel('Cantidad')
ax4.set_xticks(list(x))
ax4.set_xticklabels(categorias)
ax4.legend()
for bar in list(bars_m) + list(bars_f):
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             str(int(bar.get_height())), ha='center', va='bottom', fontsize=8)

plt.savefig('visualizacion_encuesta.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total de encuestados: {len(edades)} | Edad promedio: {sum(edades)/len(edades):.1f} años')

---
## Visualización de Resultados

A continuación se presentan gráficos generados a partir de los datos de la encuesta para facilitar el análisis visual.